# NOAA Storm Events — Split temporal y preparación segura para modelado

Esta notebook toma la base limpia generada por `1_storm_events_eda_prolija.ipynb` y prepara los conjuntos de entrenamiento, validación y test para predecir el nivel de daño económico (`DAMAGE_CLASS`).

## Objetivos

1. Revisar las variables disponibles y distinguir features, metadata y columnas prohibidas.
2. Conservar variables con nulos estructurales cuando aportan información específica de ciertos eventos.
3. Definir un split temporal reproducible.
4. Evitar leakage: ninguna transformación que aprenda estadísticas se ajusta antes del split.
5. Exportar datasets crudos para que los modelos se entrenen con pipelines reproducibles.
6. Dejar preparado un pipeline de preprocessing seguro para la siguiente notebook.

## Flujo a seguir

```text
Dataset limpio maestro
        ↓
Split temporal
        ↓
fit del preprocessing SOLO con train
        ↓
transform de train / validation / test
        ↓
modelo
```

## 1. Configuración


In [1]:
from pathlib import Path
import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# -----------------------------------------------------------------------------
# Configuración de rutas
# -----------------------------------------------------------------------------


DATA_DIR = Path('../datasets')
INPUT_PATH = DATA_DIR / 'storm_events_damage_modeling_base.csv'
OUTPUT_DIR = DATA_DIR / 'storm_splits'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Configuración del split temporal
# -----------------------------------------------------------------------------
# NOAA amplió fuertemente la cobertura de eventos a partir de 1996.
# Se usa 1996 como inicio del dataset principal de modelado, pero queda configurable.
MIN_MODEL_YEAR = 1996
TRAIN_END_YEAR = 2019
VAL_START_YEAR = 2020
VAL_END_YEAR = 2022
TEST_START_YEAR = 2023
TEST_END_YEAR = 2024

# Los años posteriores se conservan como ifnormación reciente, pero no se usan para evaluar
# el baseline porque pueden estar incompletos o sujetos a actualizaciones.
RECENT_HOLDOUT_START_YEAR = 2025

RANDOM_STATE = 42

print(f'Input:  {INPUT_PATH}')
print(f'Output: {OUTPUT_DIR}')


Input:  ../datasets/storm_events_damage_modeling_base.csv
Output: ../datasets/storm_splits


## 2. Carga y chequeos iniciales


In [2]:
storm_events_df = pd.read_csv(INPUT_PATH, low_memory=False)

print(f'Filas:    {storm_events_df.shape[0]:,}')
print(f'Columnas: {storm_events_df.shape[1]}')
print('\nColumnas disponibles:')
for idx, column in enumerate(storm_events_df.columns, start=1):
    print(f'  {idx:2d}. {column} ({storm_events_df[column].dtype})')


Filas:    1,407,565
Columnas: 44

Columnas disponibles:
   1. EVENT_ID (int64)
   2. EPISODE_ID (float64)
   3. EVENT_TYPE (str)
   4. CZ_TYPE (str)
   5. STATE (str)
   6. REGION (str)
   7. WFO (str)
   8. SOURCE (str)
   9. YEAR (int64)
  10. MONTH (int64)
  11. HOUR (int64)
  12. DAY_OF_WEEK (int64)
  13. DAY_OF_YEAR (int64)
  14. IS_WEEKEND (int64)
  15. DECADE (int64)
  16. SEASON (str)
  17. TIME_OF_DAY (str)
  18. DURATION_MIN (float64)
  19. POST_1996 (int64)
  20. BEGIN_LAT (float64)
  21. BEGIN_LON (float64)
  22. END_LAT (float64)
  23. END_LON (float64)
  24. LAT_BIN (float64)
  25. LON_BIN (float64)
  26. TRACK_DISTANCE_KM (float64)
  27. HAS_COORDINATES (int64)
  28. HAS_TRACK_DISTANCE (int64)
  29. MAGNITUDE (float64)
  30. MAGNITUDE_TYPE (str)
  31. HAS_MAGNITUDE (int64)
  32. FLOOD_CAUSE (str)
  33. TOR_SCALE_NUM (float64)
  34. TOR_LENGTH (float64)
  35. TOR_WIDTH (float64)
  36. TOR_AREA_KM2 (float64)
  37. HAS_TORNADO_DATA (int64)
  38. TOTAL_DEATHS (int64)
  39. T

In [3]:
required_columns = {
    'EVENT_ID', 'EPISODE_ID', 'YEAR', 'EVENT_TYPE', 'CZ_TYPE',
    'DAMAGE_REAL_2025', 'DAMAGE_CLASS',
}
missing_required = sorted(required_columns - set(storm_events_df.columns))
assert not missing_required, f'Faltan columnas requeridas: {missing_required}'

assert storm_events_df['EVENT_ID'].is_unique, 'EVENT_ID debe ser único.'
assert storm_events_df['DAMAGE_CLASS'].notna().all(), 'La base de modelado debe tener target completo.'
assert storm_events_df['DAMAGE_REAL_2025'].notna().all(), 'La base de modelado debe tener daño reportado.'

print('Chequeos iniciales superados.')


Chequeos iniciales superados.


## 3. Roles de las columnas

No todas las columnas exportadas por la notebook 1 deben ingresar al modelo.

### Columnas prohibidas

`DAMAGE_REAL_2025` se utiliza para construir `DAMAGE_CLASS`. Se conserva como información útil, pero nunca para entrenar.

Por ahora la información de features "humanas" también se excluyen del baseline: pueden conocerse luego del evento y darle algún tipo de fin.

### Variables con nulos estructurales

No se eliminan automáticamente:

- `TOR_SCALE_NUM`, `TOR_LENGTH`, `TOR_WIDTH`, `TOR_AREA_KM2`: aplican principalmente a tornados.
- `FLOOD_CAUSE`: aplica a inundaciones.
- `MAGNITUDE_TYPE`: describe cómo se obtuvo la magnitud de ciertos eventos de viento.
- `MAGNITUDE`: su interpretación depende del tipo de evento, pero puede aportar señal.


### `CZ_TYPE` (esto habría que borrarlo después)

Se conserva `CZ_TYPE` como variable categórica:

- `C`: county / parish.
- `Z`: forecast zone.
- `M`: marine.

No hace falta agregar `IS_MARINE` ni `IS_ZONE_FORECAST` ya que `CZ_TYPE` tiene esa info y resultarían redundantes después del OHE.


In [4]:
TARGET = 'DAMAGE_CLASS'

ID_COLUMNS = [
    'EVENT_ID',
    'EPISODE_ID',
]

# Nunca deben entrar al modelo.
FORBIDDEN_FEATURES = [
    'DAMAGE_REAL_2025',
    'TOTAL_DEATHS',
    'TOTAL_INJURIES',
    'TOTAL_CASUALTIES',
    'HAS_FATALITIES',
    'HAS_CASUALTIES',
]

# Variables categóricas nominales. Se encodearán después del split.
CATEGORICAL_FEATURES = [
    'EVENT_TYPE',
    'CZ_TYPE',
    'STATE',
    'REGION',
    'WFO',
    'SOURCE',
    'SEASON',
    'TIME_OF_DAY',
    'MAGNITUDE_TYPE',
    'FLOOD_CAUSE',
]

# Variables numéricas generales. Sus imputadores y scalers se ajustarán solo con train.
GENERAL_NUMERIC_FEATURES = [
    'YEAR',
    'MONTH',
    'HOUR',
    'DAY_OF_WEEK',
    'DAY_OF_YEAR',
    'DURATION_MIN',
    'BEGIN_LAT',
    'BEGIN_LON',
    'END_LAT',
    'END_LON',
    'LAT_BIN',
    'LON_BIN',
]

# Variables cuyo NaN suele significar “no aplica” o “no informado”.
# Se imputarán con 0, acompañadas por flags de presencia (si aplica).
STRUCTURAL_ZERO_FEATURES = [
    'TRACK_DISTANCE_KM',
    'MAGNITUDE',
    'TOR_SCALE_NUM',
    'TOR_LENGTH',
    'TOR_WIDTH',
    'TOR_AREA_KM2',
]

# Flags ya generados en notebook 1.
BINARY_FEATURES = [
    'HAS_COORDINATES',
    'HAS_TRACK_DISTANCE',
    'HAS_MAGNITUDE',
    'HAS_TORNADO_DATA',
]

# Variables derivadas redundantes que se preservan en la base inicial, pero no entran al baseline.
BASELINE_EXCLUDED_REDUNDANT = [
    'IS_WEEKEND',      # derivable de DAY_OF_WEEK
    'POST_1996',       # constante si MIN_MODEL_YEAR = 1996
    'DECADE',          # derivable de YEAR
]

# Mantener únicamente columnas realmente disponibles.
def existing(columns):
    return [column for column in columns if column in storm_events_df.columns]

ID_COLUMNS = existing(ID_COLUMNS)
FORBIDDEN_FEATURES = existing(FORBIDDEN_FEATURES)
CATEGORICAL_FEATURES = existing(CATEGORICAL_FEATURES)
GENERAL_NUMERIC_FEATURES = existing(GENERAL_NUMERIC_FEATURES)
STRUCTURAL_ZERO_FEATURES = existing(STRUCTURAL_ZERO_FEATURES)
BINARY_FEATURES = existing(BINARY_FEATURES)
BASELINE_EXCLUDED_REDUNDANT = existing(BASELINE_EXCLUDED_REDUNDANT)

BASELINE_FEATURES = (
    CATEGORICAL_FEATURES
    + GENERAL_NUMERIC_FEATURES
    + STRUCTURAL_ZERO_FEATURES
    + BINARY_FEATURES
)

print(f'Features categóricas:            {len(CATEGORICAL_FEATURES)}')
print(f'Features numéricas generales:    {len(GENERAL_NUMERIC_FEATURES)}')
print(f'Features con imputación a cero:  {len(STRUCTURAL_ZERO_FEATURES)}')
print(f'Flags binarios:                  {len(BINARY_FEATURES)}')
print(f'Total baseline:                  {len(BASELINE_FEATURES)}')


Features categóricas:            10
Features numéricas generales:    12
Features con imputación a cero:  6
Flags binarios:                  4
Total baseline:                  32


## 4. Auditoría de nulos estructurales

No se transforman los datos en esta sección. Solamente se verifica por qué ciertas variables tienen muchos faltantes y se intenta obtener interpretación correcta.

### Aclaración: Track distance

`TRACK_DISTANCE_KM` puede ser `NaN` porque faltan coordenadas o porque la distancia no aplica. No debe confundirse con una distancia realmente informada igual a `0`.

Por eso en la notebook 1 se creó:

```text
HAS_TRACK_DISTANCE = 1 → la distancia original estaba disponible
HAS_TRACK_DISTANCE = 0 → la distancia original era NaN
```

Más adelante en el pipeline se puede imputar `TRACK_DISTANCE_KM = 0`.


In [5]:
def summarize_structural_feature(df, value_col, flag_col=None):
    print('=' * 70)
    print(value_col)
    print('=' * 70)
    print(f'Nulos:             {df[value_col].isna().sum():,} ({df[value_col].isna().mean() * 100:.2f}%)')
    print(f'Ceros informados:  {df[value_col].eq(0).sum():,} ({df[value_col].eq(0).mean() * 100:.2f}%)')
    if flag_col and flag_col in df.columns:
        print(f'{flag_col}=1: {df[flag_col].eq(1).sum():,} ({df[flag_col].eq(1).mean() * 100:.2f}%)')
        print(f'Consistencia flag vs notna: {(df[flag_col].eq(1) == df[value_col].notna()).mean() * 100:.2f}%')
    print()

summarize_structural_feature(storm_events_df, 'TRACK_DISTANCE_KM', 'HAS_TRACK_DISTANCE')
summarize_structural_feature(storm_events_df, 'MAGNITUDE', 'HAS_MAGNITUDE')
summarize_structural_feature(storm_events_df, 'TOR_AREA_KM2', 'HAS_TORNADO_DATA')

for col in ['FLOOD_CAUSE', 'MAGNITUDE_TYPE', 'CZ_TYPE']:
    if col in storm_events_df.columns:
        print('=' * 70)
        print(col)
        print('=' * 70)
        print(f'Nulos: {storm_events_df[col].isna().sum():,} ({storm_events_df[col].isna().mean() * 100:.2f}%)')
        print(storm_events_df[col].value_counts(dropna=False).head(15))
        print()


TRACK_DISTANCE_KM
Nulos:             646,395 (45.92%)
Ceros informados:  560,763 (39.84%)
HAS_TRACK_DISTANCE=1: 761,170 (54.08%)
Consistencia flag vs notna: 100.00%

MAGNITUDE
Nulos:             600,665 (42.67%)
Ceros informados:  98,628 (7.01%)
HAS_MAGNITUDE=1: 806,900 (57.33%)
Consistencia flag vs notna: 100.00%

TOR_AREA_KM2
Nulos:             1,349,380 (95.87%)
Ceros informados:  2,874 (0.20%)
HAS_TORNADO_DATA=1: 229,156 (16.28%)
Consistencia flag vs notna: 87.85%

FLOOD_CAUSE
Nulos: 1,281,446 (91.04%)
FLOOD_CAUSE
NaN                             1281446
Heavy Rain                       114422
Heavy Rain / Snow Melt             5870
Heavy Rain / Tropical System       3256
Heavy Rain / Burn Area             1285
Ice Jam                             787
Dam / Levee Break                   285
Planned Dam Release                 214
Name: count, dtype: int64

MAGNITUDE_TYPE
Nulos: 990,410 (70.36%)
MAGNITUDE_TYPE
NaN    990410
EG     290913
MG     106849
E        7661
MS       7098
ES   

## 5. Definición de modelado

Para el baseline se utilizan eventos desde 1996. Los eventos previos se preservan como histórico separado.

Los años 2025 y 2026 se conservan como información reciente, pero no se usan como test principal porque pueden estar incompletos o sujetos a revisiones. De todas formas pueden llegar a utilizarse y probarse en un futuro.


In [6]:
historical_df = storm_events_df[storm_events_df['YEAR'] < MIN_MODEL_YEAR].copy()
modeling_window_df = storm_events_df[
    storm_events_df['YEAR'].between(MIN_MODEL_YEAR, TEST_END_YEAR)
].copy()
recent_holdout_df = storm_events_df[
    storm_events_df['YEAR'] >= RECENT_HOLDOUT_START_YEAR
].copy()

print(f'Histórico anterior a {MIN_MODEL_YEAR}: {len(historical_df):,} filas')
print(f'Ventana principal {MIN_MODEL_YEAR}-{TEST_END_YEAR}: {len(modeling_window_df):,} filas')
print(f'Holdout reciente desde {RECENT_HOLDOUT_START_YEAR}: {len(recent_holdout_df):,} filas')


Histórico anterior a 1996: 196,009 filas
Ventana principal 1996-2024: 1,149,478 filas
Holdout reciente desde 2025: 62,078 filas


## 6. Split temporal

La evaluación principal es temporal:

```text
Train       → 1996–2019
Validation  → 2020–2022
Test        → 2023–2024
```

Funciona mejor que un split aleatorio: el modelo aprende de eventos históricos y se evalúa sobre períodos posteriores.


In [7]:
train_df = modeling_window_df[modeling_window_df['YEAR'] <= TRAIN_END_YEAR].copy()
val_df = modeling_window_df[
    modeling_window_df['YEAR'].between(VAL_START_YEAR, VAL_END_YEAR)
].copy()
test_df = modeling_window_df[
    modeling_window_df['YEAR'].between(TEST_START_YEAR, TEST_END_YEAR)
].copy()

split_frames = {
    'train': train_df,
    'validation': val_df,
    'test': test_df,
    'recent_holdout': recent_holdout_df,
}

for split_name, split_df in split_frames.items():
    print(f'{split_name:<14}: {len(split_df):>10,} filas | '
          f'años {split_df["YEAR"].min() if len(split_df) else "-"}-'
          f'{split_df["YEAR"].max() if len(split_df) else "-"}')


train         :    876,565 filas | años 1996-2019
validation    :    156,915 filas | años 2020-2022
test          :    115,998 filas | años 2023-2024
recent_holdout:     62,078 filas | años 2025-2026


In [8]:
X_train = train_df[BASELINE_FEATURES].copy()
X_val = val_df[BASELINE_FEATURES].copy()
X_test = test_df[BASELINE_FEATURES].copy()

y_train = train_df[TARGET].copy()
y_val = val_df[TARGET].copy()
y_test = test_df[TARGET].copy()

print("Features y target separados correctamente.")
print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

Features y target separados correctamente.
X_train: (876565, 32)
X_val:   (156915, 32)
X_test:  (115998, 32)


### 6.1 Distribución del target por conjunto


In [9]:
def target_distribution(df, name):
    counts = df[TARGET].value_counts().sort_index()
    pct = (counts / len(df) * 100).round(2)
    result = pd.DataFrame({'count': counts, 'pct': pct})
    result.index.name = TARGET
    print(f'\n{name.upper()} — {len(df):,} filas')
    print(result)
    return result

train_target_dist = target_distribution(train_df, 'train')
val_target_dist = target_distribution(val_df, 'validation')
test_target_dist = target_distribution(test_df, 'test')



TRAIN — 876,565 filas
               count    pct
DAMAGE_CLASS               
0             539229  61.52
1             148246  16.91
2             126534  14.44
3              43733   4.99
4              18823   2.15

VALIDATION — 156,915 filas
               count    pct
DAMAGE_CLASS               
0             120094  76.53
1              21525  13.72
2              10959   6.98
3               3046   1.94
4               1291   0.82

TEST — 115,998 filas
              count    pct
DAMAGE_CLASS              
0             86950  74.96
1             17859  15.40
2              7736   6.67
3              2274   1.96
4              1179   1.02


## 7. Exportación de splits crudos

Los CSV se exportan **sin imputar, sin OHE y sin escalar**.

Esto permite:

- usar pipelines diferentes según el modelo;
- evitar leakage;
- auditar fácilmente los datos originales;
- comparar CatBoost con modelos que requieren OHE.

Cada split conserva metadata (`EVENT_ID`, `EPISODE_ID`) y el target.

**IMPORTANTE**: La notebook de modelado debe excluir esos identificadores antes de entrenar.


In [10]:
EXPORT_COLUMNS = list(dict.fromkeys(
    ID_COLUMNS
    + BASELINE_FEATURES
    + FORBIDDEN_FEATURES
    + [TARGET]
))
EXPORT_COLUMNS = [column for column in EXPORT_COLUMNS if column in storm_events_df.columns]

raw_split_paths = {
    'train': OUTPUT_DIR / 'storm_train_raw.csv',
    'validation': OUTPUT_DIR / 'storm_validation_raw.csv',
    'test': OUTPUT_DIR / 'storm_test_raw.csv',
    'recent_holdout': OUTPUT_DIR / 'storm_recent_holdout_raw.csv',
}

for split_name, split_df in {
    'train': train_df,
    'validation': val_df,
    'test': test_df,
    'recent_holdout': recent_holdout_df,
}.items():
    path = raw_split_paths[split_name]
    split_df[EXPORT_COLUMNS].to_csv(path, index=False)
    print(f'Exportado: {path} ({len(split_df):,} filas × {len(EXPORT_COLUMNS)} columnas)')


Exportado: ../datasets/storm_splits/storm_train_raw.csv (876,565 filas × 41 columnas)
Exportado: ../datasets/storm_splits/storm_validation_raw.csv (156,915 filas × 41 columnas)
Exportado: ../datasets/storm_splits/storm_test_raw.csv (115,998 filas × 41 columnas)
Exportado: ../datasets/storm_splits/storm_recent_holdout_raw.csv (62,078 filas × 41 columnas)


## 8. Class weights calculados únicamente con train

Los pesos de clase son parámetros del modelo. Se calculan con el conjunto de entrenamiento, no con validation ni test.


In [11]:
train_counts = train_df[TARGET].value_counts().sort_index()
n_train = len(train_df)
n_classes = train_counts.size
class_weights = (n_train / (n_classes * train_counts)).to_dict()
class_weights = {int(cls): float(weight) for cls, weight in class_weights.items()}

print('Class weights basados únicamente en train:')
for cls, weight in class_weights.items():
    print(f'  Clase {cls}: {weight:.4f}')

weights_path = OUTPUT_DIR / 'class_weights_train.json'
with open(weights_path, 'w', encoding='utf-8') as file:
    json.dump(class_weights, file, indent=2)

print(f'\nExportado: {weights_path}')


Class weights basados únicamente en train:
  Clase 0: 0.3251
  Clase 1: 1.1826
  Clase 2: 1.3855
  Clase 3: 4.0087
  Clase 4: 9.3138

Exportado: ../datasets/storm_splits/class_weights_train.json


## 9. Preprocessors según tipo de modelo

En esta sección se definen preprocessors reutilizables para la notebook de modelado.

No todos los modelos requieren el mismo tratamiento:

* Modelos lineales, SVM o KNN: OHE + escalado.
* Random Forest, Extra Trees o XGBoost: OHE sin escalado.
* CatBoost: puede trabajar con variables categóricas directamente.

Las transformaciones se ajustarán únicamente con train para evitar leakage.

Las variables se procesan de la siguiente forma:

* Categóricas: imputación con `"NotApplicable_or_Missing"` + OHE.
* Numéricas generales: imputación con la mediana de train.
* Numéricas estructurales: imputación con `0`.
* Flags binarios: se conservan sin cambios.


In [12]:
# ============================================================
# Preprocessors según tipo de modelo
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler


def build_ohe_preprocessor(scale_numeric=True):
    """
    Construye un preprocessor para modelos que requieren
    variables categóricas encodeadas con OHE.

    Parameters
    ----------
    scale_numeric : bool
        - True: escala numéricas. Útil para regresión logística,
          SVM, KNN...
        - False: conserva la escala original. Útil para modelos
          basados en árboles como Random Forest o XGBoost.
    """

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="NotApplicable_or_Missing"
                )
            ),
            (
                "ohe",
                OneHotEncoder(
                    handle_unknown="ignore",
                    min_frequency=100
                )
            ),
        ]
    )

    general_numeric_steps = [
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]

    structural_numeric_steps = [
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            )
        )
    ]

    if scale_numeric:
        general_numeric_steps.append(
            (
                "scaler",
                RobustScaler()
            )
        )

        structural_numeric_steps.append(
            (
                "scaler",
                RobustScaler()
            )
        )

    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                categorical_pipeline,
                CATEGORICAL_FEATURES
            ),
            (
                "numeric_general",
                Pipeline(steps=general_numeric_steps),
                GENERAL_NUMERIC_FEATURES
            ),
            (
                "numeric_structural",
                Pipeline(steps=structural_numeric_steps),
                STRUCTURAL_ZERO_FEATURES
            ),
            (
                "binary",
                "passthrough",
                BINARY_FEATURES
            ),
        ],
        remainder="drop",
    )


### Posibles mejoras del preprocessor

El preprocessor actual funciona como una primera versión simple y reproducible. Más adelante se pueden evaluar algunas variantes:

* Comparar OHE con otras estrategias para variables categóricas de mayor cardinalidad, como frequency encoding, target encoding controlado o modelos con soporte nativo para categóricas, como CatBoost.
* Probar distintos criterios para agrupar categorías poco frecuentes modificando `min_frequency`.
* Evaluar imputaciones alternativas para variables numéricas, por ejemplo mediana por tipo de evento en lugar de una mediana global.
* Revisar si algunas variables estructurales conviene imputarlas con `0` o mantenerlas como `NaN` según el modelo.
* Comparar escalado con `RobustScaler`, `StandardScaler` o sin escalado.
* Realizar ablations para medir si variables como `SOURCE`, `WFO`, `FLOOD_CAUSE` o las features específicas de tornados mejoran realmente el desempeño.
* Incorporar nuevamente `MAGNITUDE_ZSCORE` como experimento posterior, calculándolo únicamente con estadísticas de train.

Estas alternativas deberían compararse sobre el mismo split temporal para evitar conclusiones optimistas por leakage.


## 11. Generación opcional de datasets procesados

A partir de los splits crudos, se generan distintas versiones según el tipo de modelo:

* Modelos lineales, SVM o KNN: OHE + escalado.
* Random Forest, Extra Trees o XGBoost: OHE sin escalado.
* CatBoost: variables categóricas originales, sin OHE ni escalado.

Todas las imputaciones y transformaciones se ajustan únicamente con train para evitar leakage.

La ejecución puede activarse o desactivarse mediante flags, (procesar todas las variantes puede consumir bastante memoria).


In [13]:
# ============================================================
# 10. Generación opcional de datasets según tipo de modelo
# ============================================================

from pathlib import Path
import joblib

from scipy.sparse import save_npz


# Carpeta de salida
PROCESSED_DIR = Path("processed_datasets")
PROCESSED_DIR.mkdir(exist_ok=True)


# Elegir qué variantes generar
GENERATE_SCALED_OHE = False
GENERATE_TREE_OHE = False
GENERATE_CATBOOST = True

In [14]:

# ============================================================
# 10.1 Modelos lineales, SVM o KNN
# OHE + escalado
# ============================================================

if GENERATE_SCALED_OHE:

    print("Generando versión OHE + escalado...")

    preprocessor_scaled = build_ohe_preprocessor(
        scale_numeric=True
    )

    X_train_scaled = preprocessor_scaled.fit_transform(X_train)
    X_val_scaled = preprocessor_scaled.transform(X_val)
    X_test_scaled = preprocessor_scaled.transform(X_test)

    save_npz(
        PROCESSED_DIR / "X_train_scaled_ohe.npz",
        X_train_scaled
    )

    save_npz(
        PROCESSED_DIR / "X_val_scaled_ohe.npz",
        X_val_scaled
    )

    save_npz(
        PROCESSED_DIR / "X_test_scaled_ohe.npz",
        X_test_scaled
    )

    joblib.dump(
        preprocessor_scaled,
        PROCESSED_DIR / "preprocessor_scaled_ohe.joblib"
    )

    print("Versión OHE + escalado guardada.")
    print("Train:", X_train_scaled.shape)
    print("Validation:", X_val_scaled.shape)
    print("Test:", X_test_scaled.shape)


In [15]:
# ============================================================
# 10.2 Random Forest, Extra Trees o XGBoost
# OHE sin escalado
# ============================================================

if GENERATE_TREE_OHE:

    print("\nGenerando versión OHE sin escalado...")

    preprocessor_tree = build_ohe_preprocessor(
        scale_numeric=False
    )

    X_train_tree = preprocessor_tree.fit_transform(X_train)
    X_val_tree = preprocessor_tree.transform(X_val)
    X_test_tree = preprocessor_tree.transform(X_test)

    save_npz(
        PROCESSED_DIR / "X_train_tree_ohe.npz",
        X_train_tree
    )

    save_npz(
        PROCESSED_DIR / "X_val_tree_ohe.npz",
        X_val_tree
    )

    save_npz(
        PROCESSED_DIR / "X_test_tree_ohe.npz",
        X_test_tree
    )

    joblib.dump(
        preprocessor_tree,
        PROCESSED_DIR / "preprocessor_tree_ohe.joblib"
    )

    print("Versión OHE sin escalado guardada.")
    print("Train:", X_train_tree.shape)
    print("Validation:", X_val_tree.shape)
    print("Test:", X_test_tree.shape)


In [16]:

# ============================================================
# 10.3 CatBoost
# Categóricas nativas, sin OHE ni escalado
# ============================================================

if GENERATE_CATBOOST:

    print("\nGenerando versión para CatBoost...")

    X_train_catboost = X_train.copy()
    X_val_catboost = X_val.copy()
    X_test_catboost = X_test.copy()

    # Categóricas: completar nulos con una categoría explícita
    for col in CATEGORICAL_FEATURES:

        X_train_catboost[col] = (
            X_train_catboost[col]
            .fillna("NotApplicable_or_Missing")
            .astype(str)
        )

        X_val_catboost[col] = (
            X_val_catboost[col]
            .fillna("NotApplicable_or_Missing")
            .astype(str)
        )

        X_test_catboost[col] = (
            X_test_catboost[col]
            .fillna("NotApplicable_or_Missing")
            .astype(str)
        )

    # Numéricas generales: imputación con mediana aprendida solo con train
    for col in GENERAL_NUMERIC_FEATURES:

        median_value = X_train_catboost[col].median()

        X_train_catboost[col] = (
            X_train_catboost[col]
            .fillna(median_value)
        )

        X_val_catboost[col] = (
            X_val_catboost[col]
            .fillna(median_value)
        )

        X_test_catboost[col] = (
            X_test_catboost[col]
            .fillna(median_value)
        )

    # Numéricas estructurales: NaN -> 0
    for col in STRUCTURAL_ZERO_FEATURES:

        X_train_catboost[col] = (
            X_train_catboost[col]
            .fillna(0)
        )

        X_val_catboost[col] = (
            X_val_catboost[col]
            .fillna(0)
        )

        X_test_catboost[col] = (
            X_test_catboost[col]
            .fillna(0)
        )

    X_train_catboost.to_parquet(
        PROCESSED_DIR / "X_train_catboost.parquet",
        index=False
    )

    X_val_catboost.to_parquet(
        PROCESSED_DIR / "X_val_catboost.parquet",
        index=False
    )

    X_test_catboost.to_parquet(
        PROCESSED_DIR / "X_test_catboost.parquet",
        index=False
    )

    print("Versión para CatBoost guardada.")
    print("Train:", X_train_catboost.shape)
    print("Validation:", X_val_catboost.shape)
    print("Test:", X_test_catboost.shape)



Generando versión para CatBoost...
Versión para CatBoost guardada.
Train: (876565, 32)
Validation: (156915, 32)
Test: (115998, 32)


In [17]:
# ============================================================
# 10.4 Targets
# ============================================================

y_train.to_csv(
    PROCESSED_DIR / "y_train.csv",
    index=False
)

y_val.to_csv(
    PROCESSED_DIR / "y_val.csv",
    index=False
)

y_test.to_csv(
    PROCESSED_DIR / "y_test.csv",
    index=False
)

print("\nTargets exportados correctamente.")


Targets exportados correctamente.


## 11. Resumen de decisiones

| Tema | Decisión |
|---|---|
| Leakage de `DAMAGE_REAL_2025` | Se conserva solo como información; queda prohibido como feature. |
| Variables de tornados | Se conservan: sus nulos son estructurales y pueden aportar señal. |
| `FLOOD_CAUSE` | Se conserva como categórica; `NaN` se interpreta como no aplicable o faltante. |
| `MAGNITUDE_TYPE` | Se conserva como categórica; puede aportar señal para eventos de viento. |
| `TRACK_DISTANCE_KM` | Se conserva; `NaN` se imputa con 0 dentro del pipeline y se acompaña con `HAS_TRACK_DISTANCE`. |
| Información marina | Se conserva mediante `CZ_TYPE`; no se duplican `IS_MARINE` ni `IS_ZONE_FORECAST`. |
| Encoding | Se ajusta después del split dentro del pipeline. |
| Escalado | Se ajusta después del split dentro del pipeline. |
| Split principal | Temporal: train 1996–2019, validation 2020–2022, test 2023–2024. |
| 2025–2026 | Se preservan como holdout reciente, no como test principal del baseline. |
